In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import holidays

In [3]:
# Importando a base de dados e modificando o tipo da coluna Order Date para datetime

df = pd.read_csv('data/Superstore.csv', encoding='latin1')
datas = pd.to_datetime(df['Order Date'].copy(), format='%d-%m-%Y')
df_datas = pd.DataFrame(datas)


In [4]:
# removendo duplicatas
df_datas  = df_datas.drop_duplicates()

In [5]:
# Criando a tabela calendário (df_calendar)
data_inicio = df_datas['Order Date'].dt.year.min()
data_final = df_datas['Order Date'].dt.year.max()

df_calendar = pd.date_range(start=f'01-01-{data_inicio}', end=f'31-12-{data_final}')
df_calendar = pd.DataFrame(df_calendar,columns=['Order Date'])

In [6]:
# Dicionário contendo os meses de cada estação do ano

seasons_year = {
  'Spring':['March', 'April', 'May'], 
  'Summer':['June', 'July', 'August'], 
  'Autumn':['September', 'October', 'November'], 
  'Winter':['December','January', 'February' ]
}

In [7]:
# Criando as colunas de year, Month_num, Month_name e quarter

df_calendar['Year'] = df_calendar['Order Date'].dt.year
df_calendar['Month_num'] = df_calendar['Order Date'].dt.month
df_calendar['Month_name'] = df_calendar['Order Date'].dt.month_name()
df_calendar['Quarter'] = df_calendar['Order Date'].dt.quarter


In [8]:
# criando a coluna Season que vai conter a estação do ano que corresponde a data da venda
seasons = []

for month in df_calendar['Month_name']:
  for season, months_season in seasons_year.items():
    if month in months_season:
      seasons.append(season)

df_calendar['Season'] = seasons

In [9]:
# Criando a coluna de type_day que vai conter se o dia da data da venda é um dia útil (working day) ou fim de semana (weekend)
days = df_calendar['Order Date'].dt.weekday

df_calendar['type_day'] = np.where((days == 5) | (days == 6), 'weekend', 'working day')


In [16]:
# Adicionando a coluna Holiday. Informa se o dia é ou não um fariado
us_holidays = holidays.country_holidays('US')
is_holiday = []
holiday_name = []

for data in df_calendar['Order Date']:
  if data in us_holidays:
    is_holiday.append('Yes')
    holiday_name.append(us_holidays[data])
  else:
    is_holiday.append('No')
    holiday_name.append('--')

df_calendar['Holiday'] = is_holiday
df_calendar['Holiday Name'] = holiday_name


In [17]:
df_calendar = df_calendar.sort_values(by=['Order Date','Year','Month_num'])

In [19]:
# Exportar a base de dados
caminho = Path.cwd() / 'data'
nome_arquivo = 'calendar.csv'
caminho_completo = caminho / nome_arquivo

if Path.exists(caminho_completo):
  df_calendar.to_csv(caminho_completo, sep=',', index=False)
else:
  df_calendar.to_csv(caminho_completo, sep=',', index=False)